# Member 1 - Logistic Regression Model

## Objective
The goal of this notebook is to train and evaluate a Logistic Regression model for the Obesity dataset.


In [5]:
import os
import sys
import time
import tracemalloc

project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

from shared.preprocessing.preprocess import load_data, split_features_target, get_feature_types
from shared.evaluation.metrics import evaluate_model, print_evaluation_results

# Load the raw dataset

The raw CSV is loaded first, and then the features and target are separared.

In [6]:
file_path = os.path.join(project_root, "data", "raw", "ObesityDataSet_raw_and_data_sinthetic.csv")
df = load_data(file_path)
X, y = split_features_target(df, target_column="NObeyesdad")
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [7]:
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
numeric_features, categorical_features = get_feature_types(X)
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Numeric features: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Categorical features: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


# Build the model-specific preprocessing pipeline

For Logistic Regression, numeric features are scaled and categorical features are one-hot encoded.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=3000))
])

param_grid = {
    'model__C': [0.1, 1.0, 10.0],
    'model__solver': ['lbfgs']
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

## Train the pipeline with GridSearchCV
This follows the professor's instruction that GridSearch should be attached directly to the model pipeline.

In [9]:
start_train = time.time()
grid_search.fit(X_train, y_train)
train_time = time.time() - start_train
print('Best Parameters:', grid_search.best_params_)
print('Best Cross-Validation Score:', round(grid_search.best_score_, 4))
print(f'Training time: {train_time:.4f} seconds')

Best Parameters: {'model__C': 10.0, 'model__solver': 'lbfgs'}
Best Cross-Validation Score: 0.936


# Evaluate the best model

In [10]:
best_model = grid_search.best_estimator_
start_pred = time.time()
y_pred = best_model.predict(X_test)
pred_time = time.time() - start_pred
print(f'Prediction time: {pred_time:.4f} seconds')

tracemalloc.start()
mem_grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
mem_grid_search.fit(X_train, y_train)
_, peak_train_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

mem_best_model = mem_grid_search.best_estimator_
tracemalloc.start()
_ = mem_best_model.predict(X_test)
_, peak_pred_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'Peak memory usage during training: {peak_train_mem / 1024:.2f} KB')
print(f'Peak memory usage during prediction: {peak_pred_mem / 1024:.2f} KB')

results = evaluate_model(best_model, X_test, y_test)
results['y_pred'] = y_pred
print_evaluation_results(results)
print(f'\nTest Accuracy: {results["accuracy"]:.4f}')

Accuracy: 0.9290780141843972

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        54
           1       0.86      0.83      0.84        58
           2       0.96      0.94      0.95        70
           3       0.95      0.98      0.97        60
           4       1.00      0.98      0.99        65
           5       0.84      0.84      0.84        58
           6       0.93      0.91      0.92        58

    accuracy                           0.93       423
   macro avg       0.93      0.93      0.93       423
weighted avg       0.93      0.93      0.93       423


Confusion Matrix:
[[54  0  0  0  0  0  0]
 [ 3 48  0  0  0  7  0]
 [ 0  0 66  1  0  0  3]
 [ 0  0  1 59  0  0  0]
 [ 0  0  0  1 64  0  0]
 [ 0  8  0  0  0 49  1]
 [ 0  0  2  1  0  2 53]]

Test Accuracy: 0.9291


## Result Interception
The Logistic Regression model is used as a baseline classification model.

### Strengths
- simple and fast
- easy to train
- easy to explain
- good as a baseline model for comparison

### Weaknesses
- may not capture complex nonlinear relationships
- may perform worse than ensemble models such as Random Forest